# Nhóm 18 - Demo mô hình và bảng kết quả báo cáo tiến độ

Notebook dùng để **chạy code demo và kiểm chứng bảng kết quả**. Chọn kernel Python của project rồi **Run All**.
Không cần chạy notebook `01` trước: nếu chưa có CSV mô phỏng, notebook tự tạo bằng bộ sinh v1 đã chốt.
Nếu đã có đủ ba bảng, notebook đọc lại đúng dữ liệu đó; không sinh lại để ép kết quả khớp báo cáo.

Nội dung: chuẩn bị cặp train/dev → chạy ngẫu nhiên, luật đơn giản, lọc POC và LambdaRank → minh họa một hồ sơ →
tính lại chỉ số bằng công thức → **in bảng báo cáo ở cell cuối**. 

Các file Markdown giới thiệu phương pháp, cách đọc chỉ số và giới hạn.

**Phạm vi:** dữ liệu, nhãn và giá đều mô phỏng. Chỉ huấn luyện train, đánh giá trên 100 hồ sơ dev; không dự đoán hoặc đánh giá test T10.
Chi tiết thiết kế dữ liệu nằm trong `01_thiet_ke_du_lieu.ipynb` và `src/data.py`.


In [1]:
from pathlib import Path
import sys
from datetime import datetime
from zoneinfo import ZoneInfo
import json

import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from lightgbm import LGBMRanker

ROOT = Path.cwd()
if not (ROOT / "src/data.py").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data import (
    FEATURES, DEFAULT_DATA, generate_assets, fit_text, generate_labels,
    validate_data, save_data, load_data, build_pairs,
)
from src.experiment import MODEL_PARAMS, evaluate, group_sizes


## Chuẩn bị dữ liệu cho demo

Ba bảng `tstd/kho/gui` lần lượt chứa tài sản cần định giá, kho tài sản so sánh và các lựa chọn mô phỏng.
700 hồ sơ được chia theo thời gian: **400 train** (tháng 1–7), **100 dev** (tháng 8–9), **200 test** (tháng 10/2026).
Kho 5.000 tài sản được dùng chung, nhưng ứng viên mỗi hồ sơ phải đúng địa bàn và cửa sổ 365 ngày, không dùng tài sản tương lai/cùng báo cáo.

Mỗi cặp TSTĐ–ứng viên là một dòng học; mọi dòng cùng hồ sơ tạo thành một nhóm. Trong mỗi nhóm, ba ứng viên được chọn mang nhãn `y=1`.
Tám cột trong `FEATURES` là đầu vào, còn nhãn, mã, giá và tỷ lệ ngõ chỉ làm metadata kiểm tra/đánh giá. TF-IDF chỉ fit mô tả train.

Dữ liệu mặc định được sinh hoàn toàn từ code; không đọc dữ liệu doanh nghiệp hoặc bảng kết quả cũ. Kiểm tra tính hợp lệ trước khi học.


In [2]:
tep_du_lieu = [DEFAULT_DATA / f"{ten}.csv" for ten in ["tstd", "kho", "gui"]]
if all(p.exists() for p in tep_du_lieu):
    tstd, kho, gui = load_data(DEFAULT_DATA)
    print("Đọc lại ba bảng CSV mô phỏng đã có.")
elif any(p.exists() for p in tep_du_lieu):
    raise FileNotFoundError(
        "Bộ CSV đang thiếu bảng. Chạy python -m src.data để chuẩn bị trọn bộ."
    )
else:
    tstd, kho = generate_assets()
    gui = generate_labels(tstd, kho, fit_text(tstd))
    validate_data(tstd, kho, gui)
    save_data(tstd, kho, gui, DEFAULT_DATA)
    print("Đã tạo bộ CSV mô phỏng v1 bằng các seed đã chốt.")

thong_ke_du_lieu = validate_data(tstd, kho, gui)
vec = fit_text(tstd)
train_pairs = build_pairs(tstd, kho, gui, vec, "train")
dev_pairs = build_pairs(tstd, kho, gui, vec, "dev")

for ten, bang in [("train", train_pairs), ("dev", dev_pairs)]:
    assert not bang.isna().any().any()
    assert not bang.duplicated(["so_bc", "ma_kho"]).any()
    assert bang.groupby("so_bc")["y"].sum().eq(3).all()
    print(ten, "- số cặp:", len(bang), "- số nhóm:", bang["so_bc"].nunique(),
          "- nhãn 1:", int(bang["y"].sum()))

thong_ke_du_lieu.loc[["train", "dev"]]


Đọc lại ba bảng CSV mô phỏng đã có.


train - số cặp: 109487 - số nhóm: 400 - nhãn 1: 1200
dev - số cặp: 27546 - số nhóm: 100 - nhãn 1: 300


,so_tstd,so_cap,uv_it_nhat,uv_trung_vi,uv_nhieu_nhat
tap,,,,,
train,400,109487,250,272.0,307
dev,100,27546,254,271.5,318


## 1. Demo lập bảng kết quả trên 100 hồ sơ dev

Phần này **chạy từng phương án**, huấn luyện LambdaRank và tính lại chỉ số từ các ứng viên được chọn.

- **Câu hỏi:** LambdaRank có cải thiện Hit@3/NDCG@3 so với luật đơn giản?
- **Giả thuyết:** kết hợp 8 đặc trưng giúp học luật tương đồng có nhiễu tốt hơn thứ tự ưu tiên cố định.
- **Tiêu chí mô tả đã đặt trước:** Hit@3 cao hơn luật đơn giản; NDCG@3 không thấp hơn. Chưa kết luận ý nghĩa thống kê.
- Train: 400 hồ sơ để học. Dev: 100 hồ sơ để so sánh. **Test T10 không được dự đoán hay đánh giá**.
- Giữ nguyên dữ liệu, luật nhãn và cấu hình; không sửa seed hoặc tham số để khớp số báo cáo.

Các hàm trong `src/experiment.py` được dùng chung với CLI. Bên dưới còn có phần tính trực tiếp bằng công thức để kiểm chứng
kết quả của hàm đánh giá, và cuối notebook là bảng định dạng giống báo cáo.


In [3]:
dev_queries = tstd.loc[tstd["tap"] == "dev"].copy()
assert len(dev_queries) == 100
assert train_pairs["so_bc"].nunique() == 400
assert not set(train_pairs["so_bc"]) & set(dev_queries["so_bc"])
assert len(FEATURES) == 8
assert not set(FEATURES) & {"y", "so_bc", "ma_kho", "gia_tstd", "gia_ung_vien", "ngo_ratio"}

DEMO_DIR = ROOT / "reports/progress/notebook_demo"
DEMO_DIR.mkdir(parents=True, exist_ok=True)
protocol_demo = {
    "execution_time": datetime.now(ZoneInfo("Asia/Ho_Chi_Minh")).isoformat(),
    "evaluation_split": "dev",
    "n_dev_queries": len(dev_queries),
    "test_evaluated": False,
    "model_params": MODEL_PARAMS,
    "features": FEATURES,
    "random_baseline_seed": 8,
    "hyperparameter_search": False,
    "success_criterion": "Hit@3 hơn luật đơn giản; NDCG@3 không thấp hơn",
}
(DEMO_DIR / "protocol.json").write_text(
    json.dumps(protocol_demo, ensure_ascii=False, indent=2) + "\n"
)

thu_tu_tung_phuong_an = {}
chi_tiet_tham_chieu = {}
ket_qua_tung_phuong_an = []


def ghi_ket_qua(ten, bang_da_xep):
    # evaluate luôn giữ mẫu số đủ 100 hồ sơ, kể cả bị lọc hết ứng viên.
    chi_so, chi_tiet = evaluate(bang_da_xep, dev_queries)
    thu_tu_tung_phuong_an[ten] = bang_da_xep
    chi_tiet_tham_chieu[ten] = chi_tiet
    ket_qua_tung_phuong_an.append({"phuong_an": ten, **chi_so})
    return pd.DataFrame([chi_so], index=[ten])[
        ["n_tstd", "Hit@3", "NDCG@3", "APE_trung_vi_%", "n_APE"]
    ].round(3)


print("Cặp train:", len(train_pairs), "Cặp dev:", len(dev_pairs))
print("Tám đặc trưng:", FEATURES)
print("Chỉ đánh giá", len(dev_queries), "hồ sơ dev; chưa đánh giá test.")


Cặp train: 109487 Cặp dev: 27546
Tám đặc trưng: ['cung_duong', 'cung_nhom_kc', 'dien_tich_lech', 'mat_tien_lech', 'ngo_lech', 'kc_lech', 'do_cu_ngay', 'ct_sim']
Chỉ đánh giá 100 hồ sơ dev; chưa đánh giá test.


### 1.1 Phương án cơ sở chọn ngẫu nhiên

Mỗi ứng viên nhận một điểm ngẫu nhiên, sau đó lấy ba điểm cao nhất **trong từng hồ sơ**. Cách này không học từ train,
được dùng làm mốc để biết mức kết quả khi chưa tận dụng thông tin tương đồng.

Seed 8 cố định lượt sinh số ngẫu nhiên để có thể chạy lại. Mã kho chỉ xử lý tình huống bằng điểm, không làm đặc trưng.


In [4]:
rng_random = np.random.default_rng(8)
xep_ngau_nhien = dev_pairs.assign(
    _score=rng_random.random(len(dev_pairs))
).sort_values(
    ["so_bc", "_score", "ma_kho"],
    ascending=[True, False, True], kind="stable",
)
ghi_ket_qua("Ngẫu nhiên (seed 8)", xep_ngau_nhien)


,n_tstd,Hit@3,NDCG@3,APE_trung_vi_%,n_APE
Ngẫu nhiên (seed 8),100,2.0,0.021,22.141,100


### 1.2 Phương án cơ sở luật đơn giản

Xếp ứng viên theo thứ tự ưu tiên: **cùng đường → giống mô tả hơn → mới hơn**. Đây là thứ tự cố định do người viết đặt,
không được học từ nhãn. Giá TSTĐ và nhãn `y` không tham gia sắp xếp.

`do_cu_ngay` càng nhỏ thì dữ liệu càng mới. `ct_sim` là độ giống mô tả TF-IDF cosine, càng lớn càng giống về biểu diễn từ vựng.


In [5]:
xep_luat = dev_pairs.sort_values(
    ["so_bc", "cung_duong", "ct_sim", "do_cu_ngay", "ma_kho"],
    ascending=[True, False, False, True, True], kind="stable",
)
ghi_ket_qua("Luật đơn giản", xep_luat)


,n_tstd,Hit@3,NDCG@3,APE_trung_vi_%,n_APE
Luật đơn giản,100,43.333,0.469,12.181,100


### 1.3 Bộ lọc POC kết hợp luật đơn giản

POC ở đây là **phiên bản lọc thử nghiệm được đơn giản hóa**. Chỉ giữ ứng viên:

1. Cùng nhóm KC: cả hai có KC=0, hoặc cả hai có KC>0; KC là khoảng cách đến đường chính.
2. Độ rộng ngõ ứng viên bằng 0,7–1,5 lần ngõ TSTĐ. Ví dụ ngõ TSTĐ 4 m thì giữ ứng viên có ngõ 2,8–6 m.

Sau lọc, áp dụng đúng luật xếp hạng ở mục 1.2. Bộ lọc có thể loại cả đáp án. Vì vậy **không sinh lại nhãn sau lọc**,
không bù ứng viên, và vẫn đánh giá trên đủ 100 hồ sơ. “Trần” là tỷ lệ đáp án còn sống; “độ phủ” là tỷ lệ hồ sơ còn ít nhất ba ứng viên.


In [6]:
dieu_kien_poc = (
    dev_pairs["cung_nhom_kc"].eq(1)
    & dev_pairs["ngo_ratio"].between(0.7, 1.5)
)
dev_poc = dev_pairs.loc[dieu_kien_poc].copy()
xep_poc = dev_poc.sort_values(
    ["so_bc", "cung_duong", "ct_sim", "do_cu_ngay", "ma_kho"],
    ascending=[True, False, False, True, True], kind="stable",
)
so_uv_poc = dev_poc.groupby("so_bc").size().reindex(
    dev_queries["so_bc"], fill_value=0
)
print("Số cặp trước/sau lọc:", len(dev_pairs), len(dev_poc))
print("Đáp án còn sau lọc / tổng đáp án:", int(dev_poc["y"].sum()), "/", 3 * len(dev_queries))
print("Số hồ sơ đủ 3 ứng viên:", int(so_uv_poc.ge(3).sum()), "/", len(dev_queries))
ghi_ket_qua("Lọc POC + luật đơn giản", xep_poc)


Số cặp trước/sau lọc: 27546 4768
Đáp án còn sau lọc / tổng đáp án: 139 / 300
Số hồ sơ đủ 3 ứng viên: 99 / 100


,n_tstd,Hit@3,NDCG@3,APE_trung_vi_%,n_APE
Lọc POC + luật đơn giản,100,30.333,0.335,7.344,99


### 1.4 Mô hình LightGBM LambdaRank với 8 đặc trưng

LightGBM kết hợp nhiều cây quyết định để chấm điểm ứng viên. Mục tiêu LambdaRank hướng việc học vào **thứ tự trong từng nhóm**:
đưa các ứng viên có nhãn 1 lên những vị trí đầu.

- `X_train`: chỉ tám đặc trưng; `y_train`: nhãn lựa chọn trong train.
- `group_train`: số ứng viên mỗi TSTĐ. Các dòng cùng TSTĐ phải nằm liền nhau, tổng group bằng số dòng học.
- Giữ cấu hình đã chốt: 300 cây, learning rate 0,05, 15 lá, min_child_samples=20, seed 8; không tìm tham số, không early stopping.
- Mô hình chỉ `fit` train. Dev chỉ dùng `predict` và so sánh kết quả.

Điểm dự đoán chỉ dùng sắp thứ tự ứng viên trong nhóm; **không phải xác suất hay đơn giá**.


In [7]:
X_train = train_pairs[FEATURES]
y_train = train_pairs["y"]
group_train = group_sizes(train_pairs)

assert len(group_train) == 400
assert group_train.sum() == len(X_train)
print("X_train:", X_train.shape)
print("Số nhóm:", len(group_train), "Tổng group:", group_train.sum())

model = LGBMRanker(**MODEL_PARAMS)
model.fit(X_train, y_train, group=group_train)

# Không truyền y, giá hoặc mã vào predict.
diem_du_doan_dev = model.predict(dev_pairs[FEATURES])
xep_lambdarank = dev_pairs.assign(_score=diem_du_doan_dev).sort_values(
    ["so_bc", "_score", "ma_kho"],
    ascending=[True, False, True], kind="stable",
)
ghi_ket_qua("LambdaRank (8 đặc trưng)", xep_lambdarank)


X_train: (109487, 8)
Số nhóm: 400 Tổng group: 109487


,n_tstd,Hit@3,NDCG@3,APE_trung_vi_%,n_APE
LambdaRank (8 đặc trưng),100,57.667,0.618,8.821,100


### 1.5 Ví dụ chứng minh bằng một hồ sơ dev

Chọn **hồ sơ dev đầu tiên theo thứ tự dữ liệu**, không chọn hồ sơ theo kết quả tốt/xấu. `gui` cung cấp ba đáp án mô phỏng.
Bảng dưới liệt kê các ứng viên được mỗi phương án đưa vào top 3, thứ hạng, nhãn trúng và đơn giá mô phỏng.
Nếu một phương án còn ít hơn ba ứng viên, giữ nguyên số lượng đó để phản ánh thiếu dữ liệu.


In [8]:
ho_so_demo = dev_queries.iloc[0]
ma_ho_so_demo = ho_so_demo["so_bc"]
print("Hồ sơ:", ma_ho_so_demo)
print("Ngày:", ho_so_demo["ngay"].strftime("%d/%m/%Y"))
print("Đơn giá TSTĐ mô phỏng:", f"{ho_so_demo['gia_m2']:,.0f}", "đ/m²")
print("Ba mã đáp án:", gui.loc[
    gui["SO_BAO_CAO"].eq(ma_ho_so_demo) & gui["STT"].gt(0), "MA_KHO"
].tolist())

cac_top_demo = []
for ten, bang_xep in thu_tu_tung_phuong_an.items():
    top = bang_xep.loc[bang_xep["so_bc"].eq(ma_ho_so_demo)].head(3)
    mau = top[["ma_kho", "y", "gia_ung_vien"]].copy()
    mau.insert(0, "hang", np.arange(1, len(mau) + 1))
    mau.insert(0, "phuong_an", ten)
    cac_top_demo.append(mau)

pd.concat(cac_top_demo, ignore_index=True).rename(columns={
    "hang": "Hạng", "ma_kho": "Mã ứng viên", "y": "Trúng nhãn",
    "gia_ung_vien": "Đơn giá mô phỏng", "phuong_an": "Phương án",
})


Hồ sơ: SIM_BC_TD_00401
Ngày: 16/08/2026
Đơn giá TSTĐ mô phỏng: 40,471,163 đ/m²
Ba mã đáp án: ['SIM_KHO_04667', 'SIM_KHO_03116', 'SIM_KHO_02509']


,Phương án,Hạng,Mã ứng viên,Trúng nhãn,Đơn giá mô phỏng
0,Ngẫu nhiên (seed 8),1,SIM_KHO_02940,0,21212067.0
1,Ngẫu nhiên (seed 8),2,SIM_KHO_01775,0,19891208.0
2,Ngẫu nhiên (seed 8),3,SIM_KHO_00745,0,37056417.0
3,Luật đơn giản,1,SIM_KHO_00590,0,29040478.0
4,Luật đơn giản,2,SIM_KHO_01332,0,29282260.0
5,Luật đơn giản,3,SIM_KHO_01491,0,53116554.0
6,Lọc POC + luật đơn giản,1,SIM_KHO_01491,0,53116554.0
7,Lọc POC + luật đơn giản,2,SIM_KHO_02107,0,43003861.0
8,Lọc POC + luật đơn giản,3,SIM_KHO_04326,0,39845998.0
9,LambdaRank (8 đặc trưng),1,SIM_KHO_01332,0,29282260.0


### 1.6 Tính trực tiếp các chỉ số để kiểm chứng

Với một hồ sơ, gọi $y_1,y_2,y_3$ là nhãn của ba ứng viên theo thứ tự đề xuất:

- **Hit@3 của hồ sơ**: $(y_1+y_2+y_3)/3$. Toàn tập dùng tổng số trúng chia $3\times100$, rồi nhân 100 để báo cáo phần trăm.
- **NDCG@3**: $DCG=\sum_{i=1}^{3} y_i/\log_2(i+1)$; chia cho $IDCG=\sum_{i=1}^{3}1/\log_2(i+1)$.
  Nhãn là 0/1 nên gain bằng y. Khi lọc mất nhãn, IDCG vẫn giữ đủ ba nhãn gốc; vị trí thiếu được tính nhãn 0.
- **APE của hồ sơ (%)**: $|\overline{p}_{top3}-p_{TSTD}|/p_{TSTD}\times100$; chỉ tính khi có đủ ba ứng viên.
  Toàn tập lấy **trung vị các APE**, không lấy trung bình APE. Không có ba ứng viên thì APE là thiếu, vẫn giữ hồ sơ khi tính Hit/NDCG.

Code sau tính lại **từng hồ sơ của cả bốn phương án bằng công thức**, đối chiếu từng kết quả với hàm `evaluate`.
Đây là kiểm chứng số liệu thật từ lượt chạy, không phải dựng sẵn các giá trị trong báo cáo.


In [9]:
def tinh_truc_tiep(top, gia_tstd):
    # Chèn nhãn 0 cho vị trí còn thiếu; mẫu số của Hit luôn bằng 3.
    rel = np.zeros(3)
    rel[:len(top)] = top["y"].to_numpy()
    trong_so_vi_tri = 1 / np.log2(np.arange(2, 5))
    hit3 = rel.sum() / 3
    ndcg3 = np.dot(rel, trong_so_vi_tri) / trong_so_vi_tri.sum()
    ape = (
        abs(top["gia_ung_vien"].mean() - gia_tstd) / gia_tstd * 100
        if len(top) == 3 else np.nan
    )
    return {"so_trung": int(rel.sum()), "so_chon": len(top),
            "hit3": hit3, "ndcg3": ndcg3, "ape": ape}


chi_tiet_tu_tinh = {}
ket_qua_tu_tinh = []
for ten, bang_xep in thu_tu_tung_phuong_an.items():
    cac_nhom = {k: v for k, v in bang_xep.groupby("so_bc", sort=False)}
    cac_ho_so = []
    for _, q in dev_queries.iterrows():
        nhom = cac_nhom.get(q["so_bc"], bang_xep.iloc[:0])
        top = nhom.head(3)
        cac_ho_so.append({"so_bc": q["so_bc"], **tinh_truc_tiep(top, q["gia_m2"])})
    chi_tiet = pd.DataFrame(cac_ho_so)
    chi_tiet_tu_tinh[ten] = chi_tiet

    # So sánh từng hồ sơ, không chỉ so sánh các số trung bình cuối cùng.
    tham_chieu = chi_tiet_tham_chieu[ten].set_index("so_bc")
    for cot in ["hit3", "ndcg3", "ape"]:
        np.testing.assert_allclose(
            chi_tiet.set_index("so_bc")[cot],
            tham_chieu.reindex(chi_tiet["so_bc"])[cot],
            rtol=1e-12, atol=1e-12, equal_nan=True,
        )
    ket_qua_tu_tinh.append({
        "phuong_an": ten,
        "Hit@3": chi_tiet["so_trung"].sum() / (3 * len(dev_queries)) * 100,
        "NDCG@3": chi_tiet["ndcg3"].mean(),
        "APE_trung_vi_%": chi_tiet["ape"].median(),
        "so_dap_an_trung": int(chi_tiet["so_trung"].sum()),
        "n_APE": int(chi_tiet["ape"].notna().sum()),
    })

ket_qua_demo = pd.DataFrame(ket_qua_tu_tinh)
print("Đã đối chiếu khớp 100 hồ sơ × 4 phương án cho cả Hit@3, NDCG@3 và APE.")

# Tính tay cho hồ sơ minh họa ở mục 1.5.
pd.DataFrame([
    {"phuong_an": ten, **d.loc[d["so_bc"].eq(ma_ho_so_demo)].iloc[0].to_dict()}
    for ten, d in chi_tiet_tu_tinh.items()
])[["phuong_an", "so_chon", "so_trung", "hit3", "ndcg3", "ape"]].round(4)


Đã đối chiếu khớp 100 hồ sơ × 4 phương án cho cả Hit@3, NDCG@3 và APE.


,phuong_an,so_chon,so_trung,hit3,ndcg3,ape
0,Ngẫu nhiên (seed 8),3,0,0.0000,0.0000,35.6252
1,Luật đơn giản,3,0,0.0000,0.0000,8.2151
2,Lọc POC + luật đơn giản,3,0,0.0000,0.0000,11.9862
3,LambdaRank (8 đặc trưng),3,2,0.6667,0.5307,10.4009


### 1.7 Từ kết quả từng hồ sơ đến bảng báo cáo

Có 100 hồ sơ, mỗi hồ sơ ba đáp án: **mẫu số chung là 300 đáp án**. Bảng kiểm tra dưới đây cho biết số trúng, số hồ sơ dùng tính APE
và các chỉ số trước khi làm tròn. Nhãn bị lọc mất không được xóa khỏi mẫu số Hit@3/NDCG@3.

APE của POC có thể dùng ít hồ sơ hơn vì bộ lọc để lại dưới ba ứng viên. Vì vậy cần xem đồng thời độ phủ;
APE thấp hơn chưa đủ kết luận phương án chọn tài sản tốt hơn. Giá chỉ là giá mô phỏng, chưa áp dụng tỷ lệ điều chỉnh nghiệp vụ.


In [10]:
doi_chieu_tong = ket_qua_demo.copy()
doi_chieu_tong["mau_so_Hit@3"] = 3 * len(dev_queries)
doi_chieu_tong.rename(columns={
    "so_dap_an_trung": "Số đáp án trúng", "n_APE": "Số hồ sơ tính APE"
}).round(6)


,phuong_an,Hit@3,NDCG@3,APE_trung_vi_%,Số đáp án trúng,Số hồ sơ tính APE,mau_so_Hit@3
0,Ngẫu nhiên (seed 8),2.000000,0.021118,22.141401,6,100,300
1,Luật đơn giản,43.333333,0.468660,12.181073,130,100,300
2,Lọc POC + luật đơn giản,30.333333,0.335196,7.344146,91,99,300
3,LambdaRank (8 đặc trưng),57.666667,0.618268,8.821099,173,100,300


### 1.8 Lưu kết quả và diễn giải

Các file bên dưới là kết quả **tính trong notebook này**; không ghi đè dữ liệu hoặc luật sinh nhãn.
Lưu cả chi tiết theo hồ sơ để có thể kiểm tra lại bảng tổng hợp.

Hit@3 càng cao càng khớp lựa chọn mô phỏng; NDCG@3 càng gần 1 càng tốt về thứ tự. APE là góc nhìn giá bổ sung.
Phần nhận xét được tạo từ chính các số tính được ở trên.


In [11]:
ket_qua_demo.to_csv(DEMO_DIR / "dev_results_demo.csv", index=False, encoding="utf-8-sig")
pd.concat([
    d.assign(phuong_an=ten) for ten, d in chi_tiet_tu_tinh.items()
], ignore_index=True).to_csv(DEMO_DIR / "dev_per_query_demo.csv", index=False)

feature_importance = pd.DataFrame({
    "dac_trung": FEATURES,
    "gain": model.booster_.feature_importance(importance_type="gain"),
}).sort_values("gain", ascending=False)
feature_importance.to_csv(DEMO_DIR / "feature_importance_demo.csv", index=False)

so_lieu = ket_qua_demo.set_index("phuong_an")
hit_ml = so_lieu.loc["LambdaRank (8 đặc trưng)", "Hit@3"]
hit_luat = so_lieu.loc["Luật đơn giản", "Hit@3"]
ndcg_ml = so_lieu.loc["LambdaRank (8 đặc trưng)", "NDCG@3"]
ndcg_luat = so_lieu.loc["Luật đơn giản", "NDCG@3"]
dat_moc = hit_ml > hit_luat and ndcg_ml >= ndcg_luat

display(Markdown(
    f"**[Đo từ lượt chạy này]** LambdaRank có Hit@3 **{hit_ml:.1f}%**, "
    f"luật đơn giản **{hit_luat:.1f}%**; chênh **{hit_ml - hit_luat:.1f} điểm phần trăm**. "
    f"NDCG@3 tương ứng **{ndcg_ml:.3f}** và **{ndcg_luat:.3f}**. "
    f"Tiêu chí mô tả đặt trước: **{'ĐẠT' if dat_moc else 'CHƯA ĐẠT'}**. "
    "Kết quả chỉ áp dụng thiết lập mô phỏng; chưa đánh giá test T10."
))
print("Chi tiết lượt chạy được lưu tại:", DEMO_DIR)


**[Đo từ lượt chạy này]** LambdaRank có Hit@3 **57.7%**, luật đơn giản **43.3%**; chênh **14.3 điểm phần trăm**. NDCG@3 tương ứng **0.618** và **0.469**. Tiêu chí mô tả đặt trước: **ĐẠT**. Kết quả chỉ áp dụng thiết lập mô phỏng; chưa đánh giá test T10.

Chi tiết lượt chạy được lưu tại: /Users/phuongtran/Desktop/Studying/ML_Project/reports/progress/notebook_demo


## 2. Giới hạn và việc tiếp theo

Nhãn được sinh từ chính các đặc trưng có nhiễu; mô hình học lại quy tắc mô phỏng, chưa chứng minh khả năng lựa chọn TSSS thực tế.
Văn bản chỉ có 16 tổ hợp, địa bàn chọn đều và chưa cài đầy đủ lỗi nhập liệu. Đơn giá cũng là giả định.

Bước tiếp theo: giải thích ảnh hưởng các đặc trưng, xem các hồ sơ chọn sai, thử bỏ văn bản trên dev và khảo sát tác động của lọc POC.
Chốt cấu hình/tiêu chí trước khi đánh giá test T10 một lần cho báo cáo tiếp theo. Không sửa bộ sinh data để tăng Hit@3.

## 3. Bảng kết quả cuối

Bảng cuối lấy từ `ket_qua_demo`, tức các chỉ số tính lại trực tiếp ở mục 1.6, rồi **chỉ định dạng hiển thị**:
Hit@3 và APE một chữ số thập phân, NDCG ba chữ số, dấu phẩy thập phân theo cách viết tiếng Việt.
Tên “Ngẫu nhiên” rút gọn phần ghi seed; cách chạy vẫn là seed 8 như đã công bố.

**Lưu ý:** bảng là kết quả trên 100 hồ sơ dev. APE POC chỉ dùng hồ sơ đủ ba ứng viên, các hồ sơ còn lại vẫn tính vào
mẫu số Hit/NDCG; xem số hồ sơ cụ thể ở mục 1.7. Chưa đánh giá test và không thay giá trị để khớp ảnh.


In [12]:
ten_cot_phuong_an = f"Phương án trên {len(dev_queries)} hồ sơ dev"
bang_ket_qua_bao_cao = ket_qua_demo[
    ["phuong_an", "Hit@3", "NDCG@3", "APE_trung_vi_%"]
].rename(columns={
    "phuong_an": ten_cot_phuong_an,
    "Hit@3": "Hit@3 (%)",
    "APE_trung_vi_%": "APE trung vị (%)",
}).copy()
bang_ket_qua_bao_cao[ten_cot_phuong_an] = (
    bang_ket_qua_bao_cao[ten_cot_phuong_an]
    .replace({"Ngẫu nhiên (seed 8)": "Ngẫu nhiên"})
)

bang_hien_thi = (
    bang_ket_qua_bao_cao.style
    .hide(axis="index")
    .format({
        "Hit@3 (%)": lambda x: f"{x:.1f}".replace(".", ","),
        "NDCG@3": lambda x: f"{x:.3f}".replace(".", ","),
        "APE trung vị (%)": lambda x: f"{x:.1f}".replace(".", ","),
    })
    .set_table_styles([
        {"selector": "table", "props": [("border-collapse", "collapse"), ("font-family", "Arial")]},
        {"selector": "th", "props": [("background-color", "#17334D"), ("color", "white"),
                                       ("padding", "14px"), ("text-align", "center"), ("border", "1px solid #D9D9D9")]},
        {"selector": "td", "props": [("padding", "14px"), ("text-align", "center"),
                                       ("border", "1px solid #D9D9D9")]},
        {"selector": "tbody tr:nth-child(even)", "props": [("background-color", "#EDF5F6")]},
        {"selector": "td.col0", "props": [("text-align", "left")]},
        {"selector": "th.col0", "props": [("text-align", "left")]},
    ])
)
display(bang_hien_thi)


Phương án trên 100 hồ sơ dev,Hit@3 (%),NDCG@3,APE trung vị (%)
Ngẫu nhiên,"2,0","0,021","22,1"
Luật đơn giản,"43,3","0,469","12,2"
Lọc POC + luật đơn giản,"30,3","0,335","7,3"
LambdaRank (8 đặc trưng),"57,7","0,618","8,8"
